# データサイエンス100本ノック（構造化データ加工編） - Polars

## はじめに
- 初めに以下のセルを実行してください
- 必要なライブラリのインポートとデータベース（PostgreSQL）からのデータ読み込みを行います
- データ加工にはPolarsを使用します
- その他利用したいライブラリがあれば適宜インストールしてください（"!pip install ライブラリ名"でインストールも可能）
- 処理は複数回に分けても構いません
- 名前、住所等はダミーデータであり、実在するものではありません

In [2]:
import math  # noqa: F401
import os
from io import BytesIO
from pathlib import Path

import numpy as np  # noqa: F401
import polars as pl
from dateutil.relativedelta import relativedelta  # noqa: F401
from imblearn.under_sampling import RandomUnderSampler  # noqa: F401
from IPython.display import display  # noqa: F401
from sklearn import preprocessing  # noqa: F401
from sklearn.impute import SimpleImputer  # noqa: F401
from sklearn.model_selection import TimeSeriesSplit  # noqa: F401
from sklearn.model_selection import train_test_split  # noqa: F401
from sqlalchemy import create_engine

if "PG_PORT" in os.environ:

    host = "db"
    port = os.environ["PG_PORT"]
    database = os.environ["PG_DATABASE"]
    user = os.environ["PG_USER"]
    password = os.environ["PG_PASSWORD"]

    # PolarsでPostgreSQLから読み込むためのコネクタ
    conn = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{database}")

    df_customer = pl.read_database(query="select * from customer", connection=conn)
    df_category = pl.read_database(query="select * from category", connection=conn)
    df_product = pl.read_database(query="select * from product", connection=conn)
    df_receipt = pl.read_database(query="select * from receipt", connection=conn)
    df_store = pl.read_database(query="select * from store", connection=conn)
    df_geocode = pl.read_database(query="select * from geocode", connection=conn)

else:
    data_candidates = [
        Path("data"),
        Path("../data"),
        Path("../materials/100knocks-preprocess/docker/work/data"),
        Path("100knocks-preprocess/docker/work/data"),
    ]
    data_dir = next(
        (path for path in data_candidates if (path / "customer.csv").exists()),
        None,
    )

    if data_dir is None:
        !git clone https://github.com/The-Japan-DataScientist-Society/100knocks-preprocess
        data_dir = Path("100knocks-preprocess/docker/work/data")

    def read_csv(filename, string_columns):
        raw_data = (data_dir / filename).read_bytes()
        while raw_data.startswith(b"\xef\xbb\xbf"):
            raw_data = raw_data[3:]
        schema_overrides = {column: pl.String for column in string_columns}
        return pl.read_csv(BytesIO(raw_data), schema_overrides=schema_overrides)

    df_customer = read_csv(
        "customer.csv",
        ["customer_id", "gender_cd", "postal_cd", "application_store_cd", "status_cd"],
    )
    df_category = read_csv(
        "category.csv",
        ["category_major_cd", "category_medium_cd", "category_small_cd"],
    )
    df_product = read_csv(
        "product.csv",
        ["product_cd", "category_major_cd", "category_medium_cd", "category_small_cd"],
    )
    df_receipt = read_csv(
        "receipt.csv", ["store_cd", "customer_id", "product_cd"]
    )
    df_store = read_csv(
        "store.csv", ["store_cd", "prefecture_cd", "tel_no"]
    )
    df_geocode = read_csv("geocode.csv", ["postal_cd", "street"])

# 演習問題

---
> P-001: レシート明細データ（df_receipt）から全項目の先頭10件を表示し、どのようなデータを保有しているか目視で確認せよ。

In [ ]:
df_receipt.head(10)

---
> P-002: レシート明細データ（df_receipt）から売上年月日（sales_ymd）、顧客ID（customer_id）、商品コード（product_cd）、売上金額（amount）の順に列を指定し、10件表示せよ。

In [ ]:
df_receipt.select(['sales_ymd','customer_id','product_cd', 'amount']).head(10)

---
> P-003: レシート明細データ（df_receipt）から売上年月日（sales_ymd）、顧客ID（customer_id）、商品コード（product_cd）、売上金額（amount）の順に列を指定し、10件表示せよ。ただし、sales_ymdをsales_dateに項目名を変更して抽出すること。

In [ ]:
df_receipt.select([pl.col('sales_ymd').alias('sales_date'), 'customer_id','product_cd', 'amount']).head(10)

---
> P-004: レシート明細データ（df_receipt）から売上日（sales_ymd）、顧客ID（customer_id）、商品コード（product_cd）、売上金額（amount）の順に列を指定し、以下の条件を満たすデータを抽出せよ。
> - 顧客ID（customer_id）が"CS018205000001"

In [ ]:
df_receipt.select(['sales_ymd', 'customer_id','product_cd', 'amount']).filter(pl.col('customer_id') == 'CS018205000001')


---
> P-005: レシート明細データ（df_receipt）から売上日（sales_ymd）、顧客ID（customer_id）、商品コード（product_cd）、売上金額（amount）の順に列を指定し、以下の全ての条件を満たすデータを抽出せよ。
> - 顧客ID（customer_id）が"CS018205000001"
> - 売上金額（amount）が1,000以上

In [ ]:
df_receipt.select(['sales_ymd', 'customer_id','product_cd', 'amount']).filter((pl.col('customer_id') == 'CS018205000001') & (pl.col('amount') >= 1000))

---
> P-006: レシート明細データ（df_receipt）から売上日（sales_ymd）、顧客ID（customer_id）、商品コード（product_cd）、売上数量（quantity）、売上金額（amount）の順に列を指定し、以下の全ての条件を満たすデータを抽出せよ。
> - 顧客ID（customer_id）が"CS018205000001"
> - 売上金額（amount）が1,000以上または売上数量（quantity）が5以上

In [ ]:
df_receipt.select(['sales_ymd', 'customer_id','product_cd','quantity' ,'amount']).filter((pl.col('customer_id') == 'CS018205000001') & ((pl.col('amount') >= 1000) | (pl.col('quantity') >= 5)))

---
> P-007: レシート明細データ（df_receipt）から売上日（sales_ymd）、顧客ID（customer_id）、商品コード（product_cd）、売上金額（amount）の順に列を指定し、以下の全ての条件を満たすデータを抽出せよ。
> - 顧客ID（customer_id）が"CS018205000001"
> - 売上金額（amount）が1,000以上2,000以下

In [ ]:
df_receipt.select(['sales_ymd','customer_id', 'product_cd','amount']).filter((pl.col('customer_id') == 'CS018205000001') & (pl.col('amount').is_between(1000,2000)))

---
> P-008: レシート明細データ（df_receipt）から売上日（sales_ymd）、顧客ID（customer_id）、商品コード（product_cd）、売上金額（amount）の順に列を指定し、以下の全ての条件を満たすデータを抽出せよ。
> - 顧客ID（customer_id）が"CS018205000001"
> - 商品コード（product_cd）が"P071401019"以外

In [ ]:
df_receipt.select(['sales_ymd', 'customer_id', 'product_cd','amount']).filter((pl.col('customer_id') == 'CS018205000001') & (pl.col('product_cd') != 'P071401019'))

---
> P-009: 以下の処理において、出力結果を変えずにORをANDに書き換えよ。
> 
> `df_store.query('not(prefecture_cd == "13" | floor_area > 900)')`

In [ ]:
df_store.filter((pl.col('prefecture_cd') != '13') & (pl.col('floor_area') <= 900))

---
> P-010: 店舗データ（df_store）から、店舗コード（store_cd）が"S14"で始まるものだけ全項目抽出し、10件表示せよ。

In [ ]:
df_store.filter(pl.col('store_cd').str.starts_with('S14')).head(10)

---
> P-011: 顧客データ（df_customer）から顧客ID（customer_id）の末尾が1のものだけ全項目抽出し、10件表示せよ。

In [ ]:
df_customer.filter(pl.col('customer_id').str.ends_with('1')).head(10)

---
> P-012: 店舗データ（df_store）から、住所 (address) に"横浜市"が含まれるものだけ全項目表示せよ。

In [ ]:
df_store.filter(pl.col('address').str.contains('横浜市'))

---
> P-013: 顧客データ（df_customer）から、ステータスコード（status_cd）の先頭がアルファベットのA〜Fで始まるデータを全項目抽出し、10件表示せよ。

In [ ]:
df_customer.filter(pl.col('status_cd').str.contains(r'^[A-F]')).head(10)

---
> P-014: 顧客データ（df_customer）から、ステータスコード（status_cd）の末尾が数字の1〜9で終わるデータを全項目抽出し、10件表示せよ。

In [ ]:
df_customer.filter(pl.col('status_cd').str.contains(r'[1-9]$')).head(10)

---
> P-015: 顧客データ（df_customer）から、ステータスコード（status_cd）の先頭がアルファベットのA〜Fで始まり、末尾が数字の1〜9で終わるデータを全項目抽出し、10件表示せよ。

In [ ]:
df_customer.filter((pl.col('status_cd').str.contains(r'^[A-F]')) & (pl.col('status_cd').str.contains(r'[1-9]$'))).head(10)

#df_customer.filter(pl.col('status_cd').str.contains(r'^[A-F].*[1-9]$')).head(10)

---
> P-016: 店舗データ（df_store）から、電話番号（tel_no）が3桁-3桁-4桁のデータを全項目表示せよ。

In [ ]:
df_store.filter(pl.col("tel_no").str.contains(r"^[0-9]{3}-[0-9]{3}-[0-9]{4}$"))

---
> P-017: 顧客データ（df_customer）を生年月日（birth_day）で高齢順にソートし、先頭から全項目を10件表示せよ。

In [ ]:
df_customer.sort('birth_day').head(10)

---
> P-018: 顧客データ（df_customer）を生年月日（birth_day）で若い順にソートし、先頭から全項目を10件表示せよ。

In [ ]:
df_customer.sort('birth_day', descending=True).head(10)

---
> P-019: レシート明細データ（df_receipt）に対し、1件あたりの売上金額（amount）が高い順にランクを付与し、先頭から10件表示せよ。項目は顧客ID（customer_id）、売上金額（amount）、付与したランクを表示させること。なお、売上金額（amount）が等しい場合は同一順位を付与するものとする。

In [ ]:
df_receipt.select(
    "customer_id",
    "amount",
    pl.col("amount").rank(method="min", descending=True).alias("ranking"),
).sort("ranking").head(10)

---
> P-020: レシート明細データ（df_receipt）に対し、1件あたりの売上金額（amount）が高い順にランクを付与し、先頭から10件表示せよ。項目は顧客ID（customer_id）、売上金額（amount）、付与したランクを表示させること。なお、売上金額（amount）が等しい場合でも別順位を付与すること。

In [ ]:
df_receipt.select(
    "customer_id",
    "amount",
    pl.col("amount").rank(method="ordinal", descending=True).alias("ranking"),
).sort("ranking").head(10)

---
> P-021: レシート明細データ（df_receipt）に対し、件数をカウントせよ。

In [ ]:
len(df_receipt)

---
> P-022: レシート明細データ（df_receipt）の顧客ID（customer_id）に対し、ユニーク件数をカウントせよ。

In [ ]:
df_receipt.select(pl.col('customer_id').n_unique())

---
> P-023: レシート明細データ（df_receipt）に対し、店舗コード（store_cd）ごとに売上金額（amount）と売上数量（quantity）を合計せよ。

In [ ]:
df_receipt.group_by('store_cd').agg([pl.col('amount').sum(), pl.col('quantity').sum()]).sort('store_cd')

---
> P-024: レシート明細データ（df_receipt）に対し、顧客ID（customer_id）ごとに最も新しい売上年月日（sales_ymd）を求め、10件表示せよ。

In [ ]:
df_receipt.group_by('customer_id').agg(pl.col('sales_ymd').max()).sort('customer_id').head(10)
#group_byに入力した文字をそれぞれの要素を、ひとつのセルにリストでまとめる。

---
> P-025: レシート明細データ（df_receipt）に対し、顧客ID（customer_id）ごとに最も古い売上年月日（sales_ymd）を求め、10件表示せよ。

In [ ]:
df_receipt.group_by('customer_id').agg(pl.col('sales_ymd').min()).sort('customer_id').head(10)

---
> P-026: レシート明細データ（df_receipt）に対し、顧客ID（customer_id）ごとに最も新しい売上年月日（sales_ymd）と古い売上年月日を求め、両者が異なるデータを10件表示せよ。

In [ ]:
df_receipt.group_by('customer_id').agg([
    pl.col('sales_ymd').min().alias('sales_ymd_min'), 
    pl.col('sales_ymd').max().alias('sales_ymd_max'),
]).filter(
    pl.col('sales_ymd_min') != pl.col('sales_ymd_max')
).sort('customer_id').head(10)

---
> P-027: レシート明細データ（df_receipt）に対し、店舗コード（store_cd）ごとに売上金額（amount）の平均を計算し、降順でTOP5を表示せよ。

In [ ]:
df_receipt.group_by('store_cd').agg(pl.col('amount').mean().alias('amount_mean')).sort('amount_mean',descending=True).head(5)

---
> P-028: レシート明細データ（df_receipt）に対し、店舗コード（store_cd）ごとに売上金額（amount）の中央値を計算し、降順でTOP5を表示せよ。

In [ ]:
df_receipt.group_by("store_cd").agg(
    pl.col("amount").median().alias("amount_median")
).sort(
    ["amount_median", "store_cd"], descending=[True, False]
).head(5)

---
> P-029: レシート明細データ（df_receipt）に対し、店舗コード（store_cd）ごとに商品コード（product_cd）の最頻値を求め、10件表示させよ。

In [ ]:
df_receipt.group_by('store_cd').agg(pl.col('product_cd').mode()).sort('store_cd').head(10)

---
> P-030: レシート明細データ（df_receipt）に対し、店舗コード（store_cd）ごとに売上金額（amount）の分散を計算し、降順で5件表示せよ。

In [ ]:
df_receipt.group_by("store_cd").agg(
    pl.col("amount").var(ddof=0).alias("amount_var")
).sort("amount_var", descending=True).head(5)

---
> P-031: レシート明細データ（df_receipt）に対し、店舗コード（store_cd）ごとに売上金額（amount）の標準偏差を計算し、降順で5件表示せよ。

TIPS:

PandasとNumpyでddofのデフォルト値が異なることに注意しましょう
```
Pandas：
DataFrame.std(self, axis=None, skipna=None, level=None, ddof=1, numeric_only=None, **kwargs)
Numpy:
numpy.std(a, axis=None, dtype=None, out=None, ddof=0, keepdims=)
```

In [ ]:
df_receipt.group_by("store_cd").agg(
    pl.col("amount").std(ddof=0).alias("amount_std")
).sort("amount_std", descending=True).head(5)

---
> P-032: レシート明細データ（df_receipt）の売上金額（amount）について、25％刻みでパーセンタイル値を求めよ。

In [ ]:
df_receipt.select(
    pl.col("amount").quantile(0.25, interpolation="linear").alias("25%"),
    pl.col("amount").quantile(0.50, interpolation="linear").alias("50%"),
    pl.col("amount").quantile(0.75, interpolation="linear").alias("75%"),
    pl.col("amount").quantile(1.00, interpolation="linear").alias("100%"),
)

---
> P-033: レシート明細データ（df_receipt）に対し、店舗コード（store_cd）ごとに売上金額（amount）の平均を計算し、330以上のものを抽出せよ。

In [ ]:
df_receipt.group_by('store_cd').agg(pl.col('amount').mean().alias('amount_mean')).filter(pl.col('amount_mean') >= 330)

---
> P-034: レシート明細データ（df_receipt）に対し、顧客ID（customer_id）ごとに売上金額（amount）を合計して全顧客の平均を求めよ。ただし、顧客IDが"Z"から始まるものは非会員を表すため、除外して計算すること。

In [ ]:
df_receipt.filter(
    ~pl.col("customer_id").str.starts_with("Z")
).group_by("customer_id").agg(
    pl.col("amount").sum().alias("amount_sum")
).select(
    pl.col("amount_sum").mean().alias("amount_mean")
)

---
> P-035: レシート明細データ（df_receipt）に対し、顧客ID（customer_id）ごとに売上金額（amount）を合計して全顧客の平均を求め、平均以上に買い物をしている顧客を抽出し、10件表示せよ。ただし、顧客IDが"Z"から始まるものは非会員を表すため、除外して計算すること。

In [ ]:
df_customer_amount = df_receipt.filter(~pl.col('customer_id').str.starts_with('Z')).group_by('customer_id').agg(pl.col('amount').sum().alias('amount_sum'))
amount_mean = df_customer_amount['amount_sum'].mean()
df_customer_amount.filter(pl.col('amount_sum') >= amount_mean).sort('customer_id').head(10)


---
> P-036: レシート明細データ（df_receipt）と店舗データ（df_store）を内部結合し、レシート明細データの全項目と店舗データの店舗名（store_name）を10件表示せよ。

In [ ]:
df_receipt.join(
    df_store.select("store_cd", "store_name"),
    on="store_cd",
    how="inner",
).head(10)


---
> P-037: 商品データ（df_product）とカテゴリデータ（df_category）を内部結合し、商品データの全項目とカテゴリデータのカテゴリ小区分名（category_small_name）を10件表示せよ。

In [4]:
df_result = df_product.join(df_category.select('category_major_cd', 'category_small_name'), on='category_major_cd', how='inner')
df_result.head(10)

product_cd,category_major_cd,category_medium_cd,category_small_cd,unit_price,unit_cost,category_small_name
str,str,str,str,i64,i64,str
"""P040101001""","""04""","""0401""","""040101""",198,149,"""弁当類"""
"""P040101001""","""04""","""0401""","""040101""",198,149,"""寿司類"""
"""P040101001""","""04""","""0401""","""040101""",198,149,"""魚介佃煮類"""
"""P040101001""","""04""","""0401""","""040101""",198,149,"""海草佃煮類"""
"""P040101001""","""04""","""0401""","""040101""",198,149,"""野菜佃煮類"""
"""P040101001""","""04""","""0401""","""040101""",198,149,"""豆佃煮類"""
"""P040101001""","""04""","""0401""","""040101""",198,149,"""サラダ類"""
"""P040101001""","""04""","""0401""","""040101""",198,149,"""炒め・煮物類"""
"""P040101001""","""04""","""0401""","""040101""",198,149,"""和え物"""



---
> P-038: 顧客データ（df_customer）とレシート明細データ（df_receipt）から、顧客ごとの売上金額合計を求め、10件表示せよ。ただし、売上実績がない顧客については売上金額を0として表示させること。また、顧客は性別コード（gender_cd）が女性（1）であるものを対象とし、非会員（顧客IDが"Z"から始まるもの）は除外すること。

In [12]:
df_customer_receipt = df_customer.join(df_receipt.select('customer_id', 'amount'), on='customer_id', how='inner')
df_customer_receipt.filter((~pl.col('customer_id').str.starts_with('Z')) & (pl.col('gender_cd') == '1')).fill_null('amount').group_by('customer_id').agg(pl.col('amount').sum()).head(10)

customer_id,amount
str,i64
"""CS002613000310""",225
"""CS025613000142""",1108
"""CS029415000264""",1468
"""CS004414000181""",9584
"""CS028515000047""",2839
"""CS001515000330""",1397
"""CS002415000809""",4718
"""CS018415000125""",13400
"""CS013513000203""",786


In [6]:
df_receipt

sales_ymd,sales_epoch,store_cd,receipt_no,receipt_sub_no,customer_id,product_cd,quantity,amount
i64,i64,str,i64,i64,str,str,i64,i64
20181103,1541203200,"""S14006""",112,1,"""CS006214000001""","""P070305012""",1,158
20181118,1542499200,"""S13008""",1132,2,"""CS008415000097""","""P070701017""",1,81
20170712,1499817600,"""S14028""",1102,1,"""CS028414000014""","""P060101005""",1,170
20190205,1549324800,"""S14042""",1132,1,"""ZZ000000000000""","""P050301001""",1,25
20180821,1534809600,"""S14025""",1102,2,"""CS025415000050""","""P060102007""",1,90
…,…,…,…,…,…,…,…,…
20180221,1519171200,"""S13043""",1132,2,"""ZZ000000000000""","""P050101001""",1,40
20190911,1568160000,"""S14047""",1132,2,"""ZZ000000000000""","""P071006005""",1,218
20170311,1489190400,"""S14040""",1122,1,"""CS040513000195""","""P050405003""",1,168


---
> P-039: レシート明細データ（df_receipt）から、売上日数の多い顧客の上位20件を抽出したデータと、売上金額合計の多い顧客の上位20件を抽出したデータをそれぞれ作成し、さらにその2つを完全外部結合せよ。ただし、非会員（顧客IDが"Z"から始まるもの）は除外すること。

In [21]:
df_combined = df_receipt.filter(~pl.col('customer_id').str.starts_with('Z')).group_by('customer_id').agg(pl.col('sales_ymd').len(), pl.col('amount').sum())
df_sales_ymd_top20 = df_combined.sort('sales_ymd', descending=True).head(20)
df_amount_top20 = df_combined.sort('amount', descending=True).head(20)
df_result = df_sales_ymd_top20.join(df_amount_top20, on='customer_id', how='full')
df_result.head(40)

customer_id,sales_ymd,amount,customer_id_right,sales_ymd_right,amount_right
str,u32,i64,str,u32,i64
"""CS017415000097""",40,23086,"""CS017415000097""",40,23086
"""CS015415000185""",44,20153,"""CS015415000185""",44,20153
"""CS031414000051""",38,19202,"""CS031414000051""",38,19202
"""CS028415000007""",42,19127,"""CS028415000007""",42,19127
null,null,null,"""CS001605000009""",18,18925
…,…,…,…,…,…
"""CS021515000172""",38,13974,null,null,null
"""CS032415000209""",38,10356,null,null,null
"""CS014214000023""",38,8405,null,null,null


In [22]:
# 非会員を除外
df_members = df_receipt.filter(
    ~pl.col("customer_id").str.starts_with("Z")
)

# 売上日数が多い顧客 上位20件
df_sales_days = (
    df_members
    .group_by("customer_id")
    .agg(
        pl.col("sales_ymd")
        .n_unique()
        .alias("sales_days")
    )
    .sort("sales_days", descending=True)
    .head(20)
)

# 売上金額合計が多い顧客 上位20件
df_amount = (
    df_members
    .group_by("customer_id")
    .agg(
        pl.col("amount")
        .sum()
        .alias("amount_sum")
    )
    .sort("amount_sum", descending=True)
    .head(20)
)

# 完全外部結合
df_sales_days.join(
    df_amount,
    on="customer_id",
    how="full",
    coalesce=True,
)


customer_id,sales_days,amount_sum
str,u32,i64
"""CS017415000097""",20,23086
"""CS015415000185""",22,20153
"""CS031414000051""",19,19202
"""CS028415000007""",21,19127
"""CS001605000009""",null,18925
…,…,…
"""CS021515000172""",19,null
"""CS010214000002""",21,null
"""CS014214000023""",19,null


---
> P-040: 全ての店舗と全ての商品を組み合わせたデータを作成したい。店舗データ（df_store）と商品データ（df_product）を直積し、件数を計算せよ。

In [33]:
df_store_product = df_store.join(
    df_product,
    how='cross'
)
df_store_product.n_unique()

531590

---
> P-041: レシート明細データ（df_receipt）の売上金額（amount）を日付（sales_ymd）ごとに集計し、前回売上があった日からの売上金額増減を計算せよ。そして結果を10件表示せよ。

In [39]:
df_daily = df_receipt.group_by('sales_ymd').agg(pl.col('amount').sum().alias('amount_sum')).sort('sales_ymd')
df_daily.with_columns(pl.col('amount_sum').diff().alias('amount_sum_diff')).head(10)

sales_ymd,amount_sum,amount_sum_diff
i64,i64,i64
20170101,33723,null
20170102,24165,-9558
20170103,27503,3338
20170104,36165,8662
20170105,37830,1665
20170106,32387,-5443
20170107,23415,-8972
20170108,24737,1322
20170109,26718,1981


---
> P-042: レシート明細データ（df_receipt）の売上金額（amount）を日付（sales_ymd）ごとに集計し、各日付のデータに対し、前回、前々回、3回前に売上があった日のデータを結合せよ。そして結果を10件表示せよ。

In [41]:
df_daily = df_receipt.group_by('sales_ymd').agg(pl.col('amount').sum().alias('amount_sum')).sort('sales_ymd')
df_daily.with_columns(pl.col('amount_sum').diff(1).alias('amount_sum_diff_-1'),pl.col('amount_sum').diff(2).alias('amount_sum_diff_-2'), pl.col('amount_sum').diff(3).alias('amount_sum_diff_-3')).head(10)

sales_ymd,amount_sum,amount_sum_diff_-1,amount_sum_diff_-2,amount_sum_diff_-3
i64,i64,i64,i64,i64
20170101,33723,null,null,null
20170102,24165,-9558,null,null
20170103,27503,3338,-6220,null
20170104,36165,8662,12000,2442
20170105,37830,1665,10327,13665
20170106,32387,-5443,-3778,4884
20170107,23415,-8972,-14415,-12750
20170108,24737,1322,-7650,-13093
20170109,26718,1981,3303,-5669


---
> P-043： レシート明細データ（df_receipt）と顧客データ（df_customer）を結合し、性別コード（gender_cd）と年代（ageから計算）ごとに売上金額（amount）を合計した売上サマリデータを作成せよ。性別コードは0が男性、1が女性、9が不明を表すものとする。
>
> ただし、項目構成は年代、女性の売上金額、男性の売上金額、性別不明の売上金額の4項目とすること（縦に年代、横に性別のクロス集計）。また、年代は10歳ごとの階級とすること。

In [42]:
print(df_receipt.columns, df_customer.columns)

['sales_ymd', 'sales_epoch', 'store_cd', 'receipt_no', 'receipt_sub_no', 'customer_id', 'product_cd', 'quantity', 'amount'] ['customer_id', 'customer_name', 'gender_cd', 'gender', 'birth_day', 'age', 'postal_cd', 'address', 'application_store_cd', 'application_date', 'status_cd']


In [62]:
df_summary = df_receipt.join(
    df_customer,on='customer_id',how='inner'
    ).with_columns(((pl.col('age') // 10) * 10).alias('age_group')).group_by('age_group','gender_cd').agg(pl.col('amount').sum().alias('amount_sum'))
df_sales_summary = df_summary.pivot(on='gender_cd',index='age_group', values='amount_sum').rename({'0':'male', '1':'female', '9':'unknown'}).sort('age_group').fill_null(0)
df_sales_summary

age_group,female,unknown,male
i64,i64,i64,i64
10,149836,4317,1591
20,1363724,44328,72940
30,693047,50441,177322
40,9320791,483512,19355
50,6685192,342923,54320
60,987741,71418,272469
70,29764,2427,13435
80,262923,5111,46360
90,6260,0,0


---
> P-044： 043で作成した売上サマリデータ（df_sales_summary）は性別の売上を横持ちさせたものであった。このデータから性別を縦持ちさせ、年代、性別コード、売上金額の3項目に変換せよ。ただし、性別コードは男性を"00"、女性を"01"、不明を"99"とする。

In [71]:
df_sales_summary.unpivot(index='age_group', on=['male', 'female', 'unknown'], variable_name='gender', value_name='amount').with_columns(pl.col('gender').replace({'male':'00','female':'01','unknown':'99'}).alias('gender_cd')).drop('gender').select('age_group','gender_cd','amount').sort('age_group','gender_cd','amount')

age_group,gender_cd,amount
i64,str,i64
10,"""00""",1591
10,"""01""",149836
10,"""99""",4317
20,"""00""",72940
20,"""01""",1363724
…,…,…
80,"""01""",262923
80,"""99""",5111
90,"""00""",0


---
> P-045: 顧客データ（df_customer）の生年月日（birth_day）は日付型でデータを保有している。これをYYYYMMDD形式の文字列に変換し、顧客ID（customer_id）とともに10件表示せよ。

In [77]:
df_customer.select('customer_id',pl.col('birth_day').str.replace_all('-','').alias('birth_day')).head(2)

customer_id,birth_day
str,str
"""CS021313000114""","""19810429"""
"""CS037613000071""","""19520401"""


In [92]:
import os
import polars as pl

dtypes = {
    'customer_id': str,
    'gender_cd': str,
    'postal_cd': str,
    'application_store_cd': str,
    'status_cd': str,
    'category_major_cd': str,
    'category_medium_cd': str,
    'category_small_cd': str,
    'product_cd': str,
    'store_cd': str,
    'prefecture_cd': str,
    'tel_no': str,
    'postal_cd': str,
    'street': str,
    'application_date': str,
    'birth_day': pl.Date
}

df_customer = pl.read_csv("data/customer.csv", schema_overrides=dtypes)
df_category = pl.read_csv("data/category.csv", schema_overrides=dtypes)
df_product = pl.read_csv("data/product.csv", schema_overrides=dtypes)
df_receipt = pl.read_csv("data/receipt.csv", schema_overrides=dtypes)
df_store = pl.read_csv("data/store.csv", schema_overrides=dtypes)
df_geocode = pl.read_csv("data/geocode.csv", schema_overrides=dtypes)


---
> P-046: 顧客データ（df_customer）の申し込み日（application_date）はYYYYMMDD形式の文字列型でデータを保有している。これを日付型に変換し、顧客ID（customer_id）とともに10件表示せよ。

In [94]:
df_customer.select('customer_id', pl.col('application_date').str.to_date('%Y%m%d')).head(10)

customer_id,application_date
str,date
"""CS021313000114""",2015-09-05
"""CS037613000071""",2015-04-14
"""CS031415000172""",2015-05-29
"""CS028811000001""",2016-01-15
"""CS001215000145""",2017-06-05
"""CS020401000016""",2015-02-25
"""CS015414000103""",2015-07-22
"""CS029403000008""",2015-05-15
"""CS015804000004""",2015-06-07


customer_id,application_date
str,date
"""CS021313000114""",2015-09-05
"""CS037613000071""",2015-04-14
"""CS031415000172""",2015-05-29
"""CS028811000001""",2016-01-15
"""CS001215000145""",2017-06-05
"""CS020401000016""",2015-02-25
"""CS015414000103""",2015-07-22
"""CS029403000008""",2015-05-15
"""CS015804000004""",2015-06-07


---
> P-047: レシート明細データ（df_receipt）の売上日（sales_ymd）はYYYYMMDD形式の数値型でデータを保有している。これを日付型に変換し、レシート番号（receipt_no）、レシートサブ番号（receipt_sub_no）とともに10件表示せよ。

In [98]:
df_customer.select(
    'customer_id', 
    pl.col('application_date').cast(pl.Utf8).str.strptime(pl.Date, '%Y%m%d')).head(10)

customer_id,application_date
str,date
"""CS021313000114""",2015-09-05
"""CS037613000071""",2015-04-14
"""CS031415000172""",2015-05-29
"""CS028811000001""",2016-01-15
"""CS001215000145""",2017-06-05
"""CS020401000016""",2015-02-25
"""CS015414000103""",2015-07-22
"""CS029403000008""",2015-05-15
"""CS015804000004""",2015-06-07


---
> P-048: レシート明細データ（df_receipt）の売上エポック秒（sales_epoch）は数値型のUNIX秒でデータを保有している。これを日付型に変換し、レシート番号(receipt_no)、レシートサブ番号（receipt_sub_no）とともに10件表示せよ。

In [106]:
df_receipt.select([
    pl.from_epoch('sales_epoch', time_unit='s').dt.date().alias('sales_date'),
    'receipt_no',
    'receipt_sub_no',
]).head(10)

sales_date,receipt_no,receipt_sub_no
date,i64,i64
2018-11-03,112,1
2018-11-18,1132,2
2017-07-12,1102,1
2019-02-05,1132,1
2018-08-21,1102,2
2019-06-05,1112,1
2018-12-05,1102,2
2019-09-22,1102,1
2017-05-04,1112,2


---
> P-049: レシート明細データ（df_receipt）の売上エポック秒（sales_epoch）を日付型に変換し、「年」だけ取り出してレシート番号(receipt_no)、レシートサブ番号（receipt_sub_no）とともに10件表示せよ。

In [110]:
df_receipt.select([
    pl.from_epoch('sales_epoch', time_unit='s').dt.year().alias('sales_date'),
    'receipt_no',
    'receipt_sub_no',
]).head(10)

sales_date,receipt_no,receipt_sub_no
i32,i64,i64
2018,112,1
2018,1132,2
2017,1102,1
2019,1132,1
2018,1102,2
2019,1112,1
2018,1102,2
2019,1102,1
2017,1112,2


---
> P-050: レシート明細データ（df_receipt）の売上エポック秒（sales_epoch）を日付型に変換し、「月」だけ取り出してレシート番号(receipt_no)、レシートサブ番号（receipt_sub_no）とともに10件表示せよ。なお、「月」は0埋め2桁で取り出すこと。

In [114]:
df_receipt.select([
    pl.from_epoch('sales_epoch', time_unit='s').dt.to_string('%m').alias('sales_date'),
    'receipt_no',
    'receipt_sub_no',
]).head(10)

sales_date,receipt_no,receipt_sub_no
str,i64,i64
"""11""",112,1
"""11""",1132,2
"""07""",1102,1
"""02""",1132,1
"""08""",1102,2
"""06""",1112,1
"""12""",1102,2
"""09""",1102,1
"""05""",1112,2


---
> P-051: レシート明細データ（df_receipt）の売上エポック秒を日付型に変換し、「日」だけ取り出してレシート番号(receipt_no)、レシートサブ番号（receipt_sub_no）とともに10件表示せよ。なお、「日」は0埋め2桁で取り出すこと。

In [115]:
df_receipt.select([
    pl.from_epoch('sales_epoch', time_unit='s').dt.to_string('%d').alias('sales_date'),
    'receipt_no',
    'receipt_sub_no',
]).head(10)

sales_date,receipt_no,receipt_sub_no
str,i64,i64
"""03""",112,1
"""18""",1132,2
"""12""",1102,1
"""05""",1132,1
"""21""",1102,2
"""05""",1112,1
"""05""",1102,2
"""22""",1102,1
"""04""",1112,2


---
> P-052: レシート明細データ（df_receipt）の売上金額（amount）を顧客ID（customer_id）ごとに合計の上、売上金額合計に対して2,000円以下を0、2,000円より大きい金額を1に二値化し、顧客ID、売上金額合計とともに10件表示せよ。ただし、顧客IDが"Z"から始まるのものは非会員を表すため、除外して計算すること。

In [120]:
df_receipt.filter(
    ~pl.col('customer_id').str.starts_with('Z')
).group_by('customer_id').agg(
    pl.col('amount').sum()
).select([
    'customer_id',
    'amount',
    pl.when(pl.col('amount') > 2000).then(1).otherwise(0).alias('sales_flg')
]).sort('customer_id').head(10)

customer_id,amount,sales_flg
str,i64,i32
"""CS001113000004""",1298,0
"""CS001114000005""",626,0
"""CS001115000010""",3044,1
"""CS001205000004""",1988,0
"""CS001205000006""",3337,1
"""CS001211000025""",456,0
"""CS001212000027""",448,0
"""CS001212000031""",296,0
"""CS001212000046""",228,0


---
> P-053: 顧客データ（df_customer）の郵便番号（postal_cd）に対し、東京（先頭3桁が100〜209のもの）を1、それ以外のものを0に二値化せよ。さらにレシート明細データ（df_receipt）と結合し、全期間において売上実績のある顧客数を、作成した二値ごとにカウントせよ。

---
> P-054: 顧客データ（df_customer）の住所（address）は、埼玉県、千葉県、東京都、神奈川県のいずれかとなっている。都道府県毎にコード値を作成し、顧客ID、住所とともに10件表示せよ。値は埼玉県を11、千葉県を12、東京都を13、神奈川県を14とすること。

---
> P-055: レシート明細（df_receipt）データの売上金額（amount）を顧客ID（customer_id）ごとに合計し、その合計金額の四分位点を求めよ。その上で、顧客ごとの売上金額合計に対して以下の基準でカテゴリ値を作成し、顧客ID、売上金額合計とともに10件表示せよ。カテゴリ値は順に1〜4とする。
>
> - 最小値以上第1四分位未満 ・・・ 1を付与
> - 第1四分位以上第2四分位未満 ・・・ 2を付与
> - 第2四分位以上第3四分位未満 ・・・ 3を付与
> - 第3四分位以上 ・・・ 4を付与

---
> P-056: 顧客データ（df_customer）の年齢（age）をもとに10歳刻みで年代を算出し、顧客ID（customer_id）、生年月日（birth_day）とともに10件表示せよ。ただし、60歳以上は全て60歳代とすること。年代を表すカテゴリ名は任意とする。

---
> P-057: 056の抽出結果と性別コード（gender_cd）により、新たに性別×年代の組み合わせを表すカテゴリデータを作成し、10件表示せよ。組み合わせを表すカテゴリの値は任意とする。

---
> P-058: 顧客データ（df_customer）の性別コード（gender_cd）をダミー変数化し、顧客ID（customer_id）とともに10件表示せよ。

---
> P-059: レシート明細データ（df_receipt）の売上金額（amount）を顧客ID（customer_id）ごとに合計し、売上金額合計を平均0、標準偏差1に標準化して顧客ID、売上金額合計とともに10件表示せよ。標準化に使用する標準偏差は、分散の平方根、もしくは不偏分散の平方根のどちらでも良いものとする。ただし、顧客IDが"Z"から始まるのものは非会員を表すため、除外して計算すること。

TIPS:
- query()の引数engineで'python'か'numexpr'かを選択でき、デフォルトはインストールされていればnumexprが、無ければpythonが使われます。さらに、文字列メソッドはengine='python'でないとquery()内で使えません。


---
> P-060: レシート明細データ（df_receipt）の売上金額（amount）を顧客ID（customer_id）ごとに合計し、売上金額合計を最小値0、最大値1に正規化して顧客ID、売上金額合計とともに10件表示せよ。ただし、顧客IDが"Z"から始まるのものは非会員を表すため、除外して計算すること。

---
> P-061: レシート明細データ（df_receipt）の売上金額（amount）を顧客ID（customer_id）ごとに合計し、売上金額合計を常用対数化（底10）して顧客ID、売上金額合計とともに10件表示せよ。ただし、顧客IDが"Z"から始まるのものは非会員を表すため、除外して計算すること。

---
> P-062: レシート明細データ（df_receipt）の売上金額（amount）を顧客ID（customer_id）ごとに合計し、売上金額合計を自然対数化（底e）して顧客ID、売上金額合計とともに10件表示せよ。ただし、顧客IDが"Z"から始まるのものは非会員を表すため、除外して計算すること。

---
> P-063: 商品データ（df_product）の単価（unit_price）と原価（unit_cost）から各商品の利益額を算出し、結果を10件表示せよ。

---
> P-064: 商品データ（df_product）の単価（unit_price）と原価（unit_cost）から、各商品の利益率の全体平均を算出せよ。ただし、単価と原価には欠損が生じていることに注意せよ。

---
> P-065: 商品データ（df_product）の各商品について、利益率が30%となる新たな単価を求めよ。ただし、1円未満は切り捨てること。そして結果を10件表示させ、利益率がおよそ30％付近であることを確認せよ。ただし、単価（unit_price）と原価（unit_cost）には欠損が生じていることに注意せよ。

---
> P-066: 商品データ（df_product）の各商品について、利益率が30%となる新たな単価を求めよ。今回は、1円未満を丸めること（四捨五入または偶数への丸めで良い）。そして結果を10件表示させ、利益率がおよそ30％付近であることを確認せよ。ただし、単価（unit_price）と原価（unit_cost）には欠損が生じていることに注意せよ。

---
> P-067: 商品データ（df_product）の各商品について、利益率が30%となる新たな単価を求めよ。今回は、1円未満を切り上げること。そして結果を10件表示させ、利益率がおよそ30％付近であることを確認せよ。ただし、単価（unit_price）と原価（unit_cost）には欠損が生じていることに注意せよ。

---
> P-068: 商品データ（df_product）の各商品について、消費税率10％の税込み金額を求めよ。1円未満の端数は切り捨てとし、結果を10件表示せよ。ただし、単価（unit_price）には欠損が生じていることに注意せよ。

---
> P-069: レシート明細データ（df_receipt）と商品データ（df_product）を結合し、顧客毎に全商品の売上金額合計と、カテゴリ大区分コード（category_major_cd）が"07"（瓶詰缶詰）の売上金額合計を計算の上、両者の比率を求めよ。抽出対象はカテゴリ大区分コード"07"（瓶詰缶詰）の売上実績がある顧客のみとし、結果を10件表示せよ。

---
> P-070: レシート明細データ（df_receipt）の売上日（sales_ymd）に対し、顧客データ（df_customer）の会員申込日（application_date）からの経過日数を計算し、顧客ID（customer_id）、売上日、会員申込日とともに10件表示せよ（sales_ymdは数値、application_dateは文字列でデータを保持している点に注意）。

---
> P-071: レシート明細データ（df_receipt）の売上日（sales_ymd）に対し、顧客データ（df_customer）の会員申込日（application_date）からの経過月数を計算し、顧客ID（customer_id）、売上日、会員申込日とともに10件表示せよ（sales_ymdは数値、application_dateは文字列でデータを保持している点に注意）。1ヶ月未満は切り捨てること。

---
> P-072: レシート明細データ（df_receipt）の売上日（df_customer）に対し、顧客データ（df_customer）の会員申込日（application_date）からの経過年数を計算し、顧客ID（customer_id）、売上日、会員申込日とともに10件表示せよ（sales_ymdは数値、application_dateは文字列でデータを保持している点に注意）。1年未満は切り捨てること。

---
> P-073: レシート明細データ（df_receipt）の売上日（sales_ymd）に対し、顧客データ（df_customer）の会員申込日（application_date）からのエポック秒による経過時間を計算し、顧客ID（customer_id）、売上日、会員申込日とともに10件表示せよ（なお、sales_ymdは数値、application_dateは文字列でデータを保持している点に注意）。なお、時間情報は保有していないため各日付は0時0分0秒を表すものとする。

---
> P-074: レシート明細データ（df_receipt）の売上日（sales_ymd）に対し、当該週の月曜日からの経過日数を計算し、売上日、直前の月曜日付とともに10件表示せよ（sales_ymdは数値でデータを保持している点に注意）。

---
> P-075: 顧客データ（df_customer）からランダムに1%のデータを抽出し、先頭から10件表示せよ。

---
> P-076: 顧客データ（df_customer）から性別コード（gender_cd）の割合に基づきランダムに10%のデータを層化抽出し、性別コードごとに件数を集計せよ。

---
> P-077: レシート明細データ（df_receipt）の売上金額を顧客単位に合計し、合計した売上金額の外れ値を抽出せよ。なお、外れ値は売上金額合計を対数化したうえで平均と標準偏差を計算し、その平均から3σを超えて離れたものとする（自然対数と常用対数のどちらでも可）。結果は10件表示せよ。

---
> P-078: レシート明細データ（df_receipt）の売上金額（amount）を顧客単位に合計し、合計した売上金額の外れ値を抽出せよ。ただし、顧客IDが"Z"から始まるのものは非会員を表すため、除外して計算すること。なお、ここでは外れ値を第1四分位と第3四分位の差であるIQRを用いて、「第1四分位数-1.5×IQR」を下回るもの、または「第3四分位数+1.5×IQR」を超えるものとする。結果は10件表示せよ。

---
> P-079: 商品データ（df_product）の各項目に対し、欠損数を確認せよ。

---
> P-080: 商品データ（df_product）のいずれかの項目に欠損が発生しているレコードを全て削除した新たな商品データを作成せよ。なお、削除前後の件数を表示させ、079で確認した件数だけ減少していることも確認すること。

---
> P-081: 単価（unit_price）と原価（unit_cost）の欠損値について、それぞれの平均値で補完した新たな商品データを作成せよ。なお、平均値については1円未満を丸めること（四捨五入または偶数への丸めで良い）。補完実施後、各項目について欠損が生じていないことも確認すること。

---
> P-082: 単価（unit_price）と原価（unit_cost）の欠損値について、それぞれの中央値で補完した新たな商品データを作成せよ。なお、中央値については1円未満を丸めること（四捨五入または偶数への丸めで良い）。補完実施後、各項目について欠損が生じていないことも確認すること。

---
> P-083: 単価（unit_price）と原価（unit_cost）の欠損値について、各商品のカテゴリ小区分コード（category_small_cd）ごとに算出した中央値で補完した新たな商品データを作成せよ。なお、中央値については1円未満を丸めること（四捨五入または偶数への丸めで良い）。補完実施後、各項目について欠損が生じていないことも確認すること。

---
> P-084: 顧客データ（df_customer）の全顧客に対して全期間の売上金額に占める2019年売上金額の割合を計算し、新たなデータを作成せよ。ただし、売上実績がない場合は0として扱うこと。そして計算した割合が0超のものを抽出し、結果を10件表示せよ。また、作成したデータに欠損が存在しないことを確認せよ。

---
> P-085: 顧客データ（df_customer）の全顧客に対し、郵便番号（postal_cd）を用いてジオコードデータ（df_geocode）を紐付け、新たな顧客データを作成せよ。ただし、1つの郵便番号（postal_cd）に複数の経度（longitude）、緯度（latitude）情報が紐づく場合は、経度（longitude）、緯度（latitude）の平均値を算出して使用すること。また、作成結果を確認するために結果を10件表示せよ。

---
> P-086: 085で作成した緯度経度つき顧客データに対し、会員申込店舗コード（application_store_cd）をキーに店舗データ（df_store）と結合せよ。そして申込み店舗の緯度（latitude）・経度情報（longitude)と顧客住所（address）の緯度・経度を用いて申込み店舗と顧客住所の距離（単位：km）を求め、顧客ID（customer_id）、顧客住所（address）、店舗住所（address）とともに表示せよ。計算式は以下の簡易式で良いものとするが、その他精度の高い方式を利用したライブラリを利用してもかまわない。結果は10件表示せよ。

$$
\mbox{緯度（ラジアン）}：\phi \\
\mbox{経度（ラジアン）}：\lambda \\
\mbox{距離}L = 6371 * \arccos(\sin \phi_1 * \sin \phi_2
+ \cos \phi_1 * \cos \phi_2 * \cos(\lambda_1 − \lambda_2))
$$

---
> P-087: 顧客データ（df_customer）では、異なる店舗での申込みなどにより同一顧客が複数登録されている。名前（customer_name）と郵便番号（postal_cd）が同じ顧客は同一顧客とみなして1顧客1レコードとなるように名寄せした名寄顧客データを作成し、顧客データの件数、名寄顧客データの件数、重複数を算出せよ。ただし、同一顧客に対しては売上金額合計が最も高いものを残し、売上金額合計が同一もしくは売上実績がない顧客については顧客ID（customer_id）の番号が小さいものを残すこととする。

---
> P-088: 087で作成したデータを元に、顧客データに統合名寄IDを付与したデータを作成せよ。ただし、統合名寄IDは以下の仕様で付与するものとする。
>
> - 重複していない顧客：顧客ID（customer_id）を設定
> - 重複している顧客：前設問で抽出したレコードの顧客IDを設定
> 
> 顧客IDのユニーク件数と、統合名寄IDのユニーク件数の差も確認すること。

---
> P-089: 売上実績がある顧客を、予測モデル構築のため学習用データとテスト用データに分割したい。それぞれ8:2の割合でランダムにデータを分割せよ。

---
> P-090: レシート明細データ（df_receipt）は2017年1月1日〜2019年10月31日までのデータを有している。売上金額（amount）を月次で集計し、学習用に12ヶ月、テスト用に6ヶ月の時系列モデル構築用データを3セット作成せよ。

---
> P-091: 顧客データ（df_customer）の各顧客に対し、売上実績がある顧客数と売上実績がない顧客数が1:1となるようにアンダーサンプリングで抽出せよ。

---
> P-092: 顧客データ（df_customer）の性別について、第三正規形へと正規化せよ。

---
> P-093: 商品データ（df_product）では各カテゴリのコード値だけを保有し、カテゴリ名は保有していない。カテゴリデータ（df_category）と組み合わせて非正規化し、カテゴリ名を保有した新たな商品データを作成せよ。

---
> P-094: 093で作成したカテゴリ名付き商品データを以下の仕様でファイル出力せよ。
>
> |ファイル形式|ヘッダ有無|文字エンコーディング|
> |:--:|:--:|:--:|
> |CSV（カンマ区切り）|有り|UTF-8|
> 
> ファイル出力先のパスは以下のようにすること
> 
> |出力先|
> |:--:|
> |./data|

---
> P-095: 093で作成したカテゴリ名付き商品データを以下の仕様でファイル出力せよ。
>
> |ファイル形式|ヘッダ有無|文字エンコーディング|
> |:--:|:--:|:--:|
> |CSV（カンマ区切り）|有り|CP932|
> 
> ファイル出力先のパスは以下のようにすること。
> 
> |出力先|
> |:--:|
> |./data|

---
> P-096: 093で作成したカテゴリ名付き商品データを以下の仕様でファイル出力せよ。
>
> |ファイル形式|ヘッダ有無|文字エンコーディング|
> |:--:|:--:|:--:|
> |CSV（カンマ区切り）|無し|UTF-8|
> 
> ファイル出力先のパスは以下のようにすること。
> 
> |出力先|
> |:--:|
> |./data|

---
> P-097: 094で作成した以下形式のファイルを読み込み、データを3件を表示させて正しく取り込まれていることを確認せよ。
> 
> |ファイル形式|ヘッダ有無|文字エンコーディング|
> |:--:|:--:|:--:|
> |CSV（カンマ区切り）|有り|UTF-8|

---
> P-098: 096で作成した以下形式のファイルを読み込み、データを3件を表示させて正しく取り込まれていることを確認せよ。
> 
> |ファイル形式|ヘッダ有無|文字エンコーディング|
> |:--:|:--:|:--:|
> |CSV（カンマ区切り）|ヘッダ無し|UTF-8|

---
> P-099: 093で作成したカテゴリ名付き商品データを以下の仕様でファイル出力せよ。
>
> |ファイル形式|ヘッダ有無|文字エンコーディング|
> |:--:|:--:|:--:|
> |TSV（タブ区切り）|有り|UTF-8|
> 
> ファイル出力先のパスは以下のようにすること
> 
> |出力先|
> |:--:|
> |./data|

---
> P-100: 099で作成した以下形式のファイルを読み込み、データを3件を表示させて正しく取り込まれていることを確認せよ。
> 
> |ファイル形式|ヘッダ有無|文字エンコーディング|
> |:--:|:--:|:--:|
> |TSV（タブ区切り）|有り|UTF-8|

# これで１００本終わりです。おつかれさまでした！